# **Data Analysis**

The goal of this file is to conduct data analysis for the Los Angeles Angels (LAA) and Los Angeles Dodgers (LAD) trade, home run, and score data.

In [11]:
import random
import pandas as pd
import plotly.express as px
from datetime import timedelta
from utils.data_handlers import DataProcessing
from utils.data_analysis_helpers import plot_corr_matrix

In [ ]:
# Instantiate DataProcessing objects for LAA and LAD

laa_DP = DataProcessing(
    team = 'LAA',
    trade_path = "../data/raw/laa_kalshi_trade_data.parquet",
    score_path = "../data/raw/laa_score_data.parquet",
    homerun_path = "../data/raw/laa_homerun_data.parquet"
)

laa_trade_df = laa_DP.get_trade_df()
laa_score_df = laa_DP.get_score_df()
laa_homerun_df = laa_DP.get_homerun_df()

lad_DP = DataProcessing(
    team = 'LAD',
    trade_path = "../data/raw/lad_kalshi_trade_data.parquet",
    score_path = "../data/raw/lad_score_data.parquet",
    homerun_path = "../data/raw/lad_homerun_data.parquet"
)

lad_trade_df = lad_DP.get_trade_df()
lad_score_df = lad_DP.get_score_df()
lad_homerun_df = lad_DP.get_homerun_df()

## Graphical Analysis for LAA

In [13]:
# Merge LAA trade and home run frames

homerun_df_sorted = laa_homerun_df.copy().sort_values('time_pst').reset_index(drop=True)
trade_df_sorted = laa_trade_df.copy().sort_values('time_pst').reset_index(drop=True)

laa_hr_events = pd.merge_asof(
    homerun_df_sorted,
    trade_df_sorted,
    on = 'time_pst',
    by = 'ticker',
    direction = 'backward',
    suffixes = ('', '_at_hr'),
    tolerance = pd.Timedelta('3min')
)

laa_df = pd.merge(
    trade_df_sorted,
    laa_hr_events,
    how='outer'
)

laa_df['laa_homerun_dummy'] = laa_df['laa_homerun_dummy'].fillna(0).astype(int)
laa_df['homerun_dummy'] = laa_df['homerun_dummy'].fillna(0).astype(int)

laa_df = laa_df.dropna(subset=['price']).reset_index(drop=True)

laa_df = laa_df[[
    'ticker', 'time_pst', 'price',
    'laa_homerun_dummy', 'homerun_dummy'
]]

laa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109669 entries, 0 to 109668
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype                              
---  ------             --------------   -----                              
 0   ticker             109669 non-null  object                             
 1   time_pst           109669 non-null  datetime64[ns, America/Los_Angeles]
 2   price              109669 non-null  float64                            
 3   laa_homerun_dummy  109669 non-null  int64                              
 4   homerun_dummy      109669 non-null  int64                              
dtypes: datetime64[ns, America/Los_Angeles](1), float64(1), int64(2), object(1)
memory usage: 4.2+ MB


In [14]:
# Plot home runs 

laa_colors = ['#BA0021', '#C4CED4', '#003263']

unique_tickers = list(set(laa_df['ticker']))
random_ticker_subset = random.sample(unique_tickers, k=min(10, len(unique_tickers)))

laa_df['time_pst'] = pd.to_datetime(laa_df['time_pst'])
homerun_subset = laa_df[
    (laa_df['homerun_dummy'] == 1) & 
    (laa_df['ticker'].isin(random_ticker_subset))
].copy()

for idx, hr_event in homerun_subset.iterrows():
    hr_time = hr_event['time_pst']
    ticker = hr_event['ticker']
    is_laa = hr_event['laa_homerun_dummy'] == 1
    
    start_window = hr_time - timedelta(minutes=6)
    end_window = hr_time + timedelta(minutes=6)
    
    game_window = laa_df[
        (laa_df['ticker'] == ticker) & 
        (laa_df['time_pst'] >= start_window) & 
        (laa_df['time_pst'] <= end_window)
    ].copy()

    if game_window.empty:
        continue

    date_str = f"{ticker[12:15]} {ticker[15:17]}, 20{ticker[10:12]}"
    opp_team = ticker[17:].split('-')[0].replace('LAA', '')
    hr_type = "LAA" if is_laa else "Opponent"

    fig = px.line(game_window, x='time_pst', y='price', color_discrete_sequence=[laa_colors[0]])

    fig.update_layout(
        template='plotly_dark',
        title={
            'text': f"<b>{hr_type} Home Run</b><br><span style='font-size:16px; color:{laa_colors[1]}'>LAA vs {opp_team} {date_str}</span>",
            'font': {'size': 20, 'color': laa_colors[1]},
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Time (PST)',
        yaxis_title='Price (LAA wins)',
        hovermode='x unified',
        paper_bgcolor='#111111', 
        plot_bgcolor='#111111'
    )

    label_text = "LAA HR" if is_laa else "Opponent HR"
    border_color = laa_colors[1] if is_laa else laa_colors[2]
    bg_color = 'rgba(186, 0, 33, 0.9)' if is_laa else 'rgba(0, 50, 99, 0.7)'

    fig.add_annotation(
        x=hr_time,
        y=hr_event['price'],
        text=f"<b>{label_text}</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor='white', 
        ax=0,
        ay=60,         
        font=dict(color='white', size=11),
        bgcolor=bg_color,
        bordercolor=border_color,
        borderwidth=2,
        xanchor='center',
        yanchor='top' 
    )

    fig.show()

## Graphical Analysis for LAD

In [15]:
# Merge LAD trade and home run frames

homerun_df_sorted = lad_homerun_df.copy().sort_values('time_pst').reset_index(drop=True)
trade_df_sorted = lad_trade_df.copy().sort_values('time_pst').reset_index(drop=True)

lad_hr_events = pd.merge_asof(
    homerun_df_sorted,
    trade_df_sorted,
    on = 'time_pst',
    by = 'ticker',
    direction = 'backward',
    suffixes = ('', '_at_hr'),
    tolerance = pd.Timedelta('3min')
)

lad_df = pd.merge(
    trade_df_sorted,
    lad_hr_events,
    how='outer'
)

lad_df['lad_homerun_dummy'] = lad_df['lad_homerun_dummy'].fillna(0).astype(int)
lad_df['homerun_dummy'] = lad_df['homerun_dummy'].fillna(0).astype(int)

lad_df = lad_df.dropna(subset=['price']).reset_index(drop=True)

lad_df = lad_df[[
    'ticker', 'time_pst', 'price',
    'lad_homerun_dummy', 'homerun_dummy'
]]

lad_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159729 entries, 0 to 159728
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype                              
---  ------             --------------   -----                              
 0   ticker             159729 non-null  object                             
 1   time_pst           159729 non-null  datetime64[ns, America/Los_Angeles]
 2   price              159729 non-null  float64                            
 3   lad_homerun_dummy  159729 non-null  int64                              
 4   homerun_dummy      159729 non-null  int64                              
dtypes: datetime64[ns, America/Los_Angeles](1), float64(1), int64(2), object(1)
memory usage: 6.1+ MB


In [16]:
# Plot home runs 

lad_colors = ['#EF3E42', '#A5ACAF', '#005A9C']

unique_tickers = list(set(lad_df['ticker']))
random_ticker_subset = random.sample(unique_tickers, k=min(10, len(unique_tickers)))

lad_df['time_pst'] = pd.to_datetime(lad_df['time_pst'])

homerun_subset = lad_df[
    (lad_df['homerun_dummy'] == 1) & 
    (lad_df['ticker'].isin(random_ticker_subset))
].copy()

for idx, hr_event in homerun_subset.iterrows():
    hr_time = hr_event['time_pst']
    ticker = hr_event['ticker']
    is_lad = hr_event['lad_homerun_dummy'] == 1
    
    start_window = hr_time - timedelta(minutes=6)
    end_window = hr_time + timedelta(minutes=6)
    
    game_window = lad_df[
        (lad_df['ticker'] == ticker) & 
        (lad_df['time_pst'] >= start_window) & 
        (lad_df['time_pst'] <= end_window)
    ].copy()

    if game_window.empty:
        continue

    date_str = f"{ticker[12:15]} {ticker[15:17]}, 20{ticker[10:12]}"
    opp_team = ticker[17:].split('-')[0].replace('LAD', '')
    hr_type = "LAD" if is_lad else "Opponent"

    fig = px.line(game_window, x='time_pst', y='price', color_discrete_sequence=[lad_colors[2]])

    fig.update_layout(
        template='plotly_dark',
        title={
            'text': f"<b>{hr_type} Home Run</b><br><span style='font-size:16px; color:{lad_colors[1]}'>LAD vs {opp_team} {date_str}</span>",
            'font': {'size': 20, 'color': 'white'},
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Time (PST)',
        yaxis_title='Price (LAD wins)',
        hovermode='x unified',
        paper_bgcolor='#0a0a0a', 
        plot_bgcolor='#0a0a0a'
    )

    label_text = "LAD HR" if is_lad else "Opponent HR"
    border_color = lad_colors[0] if is_lad else lad_colors[1]
    bg_color = 'rgba(0, 90, 156, 0.9)' if is_lad else 'rgba(60, 60, 60, 0.8)'

    fig.add_annotation(
        x=hr_time,
        y=hr_event['price'],
        text=f"<b>{label_text}</b>",
        showarrow=True,
        arrowhead=2,
        arrowcolor='white',
        ax=0,
        ay=60, 
        font=dict(color='white', size=11),
        bgcolor=bg_color,
        bordercolor=border_color,
        borderwidth=2,
        xanchor='center',
        yanchor='top' 
    )

    fig.show()

### Observations

- Generally, home runs seem to occur near large price movements, but they do not always align with what would be expected. The theory is that a home run occurs prior to a large directional price movement:

    - If the opponent hits a home run, then the price goes down.

    - If LAA/LAD hits a home run, then the price goes up.

- Also, home runs do not always lead to large price movements.


### Model Implications

- Generic Price Changes

    - Just look at the price for the next trade.

- Forward Binned Mean Price Changes

    - Define a window parameter $w$ that determines how many trades to skip after the home run event.

    - Define a bin parameter $b$ that determines how many trades to include in the forward bin.
    
    - The forward bin includes $b$ trades starting from $w$ trades after the home run: indices $[i + w, i + w + b - 1]$
    
    - The absolute forward price change is defined as: $$p_{f}(w,b) = \bar{p}_{forward}(w,b) - p_{hr}$$

    - The percent forward price change is defined as: $$\tilde{p}_{f}(w,b) = \frac{\bar{p}_{forward}(w,b) - p_{hr}}{p_{hr}}$$

        - Where $\bar{p}_{forward}(w,b)$ is the average price in the forward bin containing $b$ trades, and $p_{hr}$ is the price at the home run event.

- Double Binned Mean Price Changes

    - Assume the home run data (from `mlbstatsapi`) is slightly delayed.

    - Define a window parameter $w$ that determines how many trades to skip on either side of the recorded home run event.

    - Define a bin parameter $b$ that determines how many trades to include in each bin.
    
    - The "before" bin includes $b$ trades starting from $w$ trades before the home run: indices $[i - w - b + 1, i - w]$
    
    - The "after" bin includes $b$ trades starting from $w$ trades after the home run: indices $[i + w, i + w + b - 1]$
    
    - The absolute price change is defined as: $$p_{d}(w,b) = \bar{p}_{after}(w,b) - \bar{p}_{before}(w,b)$$

    - The percent price change is defined as: $$\tilde{p}_{d}(w,b) = \frac{\bar{p}_{after}(w,b) - \bar{p}_{before}(w,b)}{\bar{p}_{before}(w,b)}$$

        - Where $\bar{p}_{after}(w,b)$ is the average price in the "after" bin and $\bar{p}_{before}(w,b)$ is the average price in the "before" bin.
        
        - Both bins contain exactly $b$ trades each, positioned $w$ trades away from the home run event.

    - Additionally, the double binned mean price can be defined as:$$\bar{p}_{before}(w,b)$$

## Statistical Analysis for LAA

In [17]:
# Load in feature engineered data frames

laa_generic_df = pd.read_parquet("../data/processed/laa_generic_data.parquet")
laa_forward_df = pd.read_parquet("../data/processed/laa_forward_data.parquet")
laa_double_df = pd.read_parquet("../data/processed/laa_double_data.parquet")

### Generic Data Frame Analysis

In [18]:
# Define features and targets

generic_features = [
    'price', 'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

generic_targets = ['g_pct_px_chg']

# Look at a subset of the features, and drop any price change data if missing
laa_generic_df = laa_generic_df[generic_targets + generic_features].dropna(subset=['g_pct_px_chg'])

laa_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 328 entries, 0 to 350
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_pct_px_chg       328 non-null    float64
 1   price              328 non-null    float64
 2   laa_homerun_dummy  328 non-null    int64  
 3   runners_on_base    328 non-null    float64
 4   inning             328 non-null    float64
 5   laa_score          328 non-null    float64
 6   opp_score          328 non-null    float64
 7   score_delta        328 non-null    float64
 8   score_delta_abs    328 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 25.6 KB


In [19]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_generic_df[laa_generic_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[generic_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_generic_df[laa_generic_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[generic_targets].describe(), 2))



LAA Home Runs:
       g_pct_px_chg
count        172.00
mean           0.17
std            0.26
min           -0.33
25%            0.00
50%            0.09
75%            0.26
max            1.36

Opponent Home Runs:
       g_pct_px_chg
count        156.00
mean          -0.18
std            0.27
min           -0.86
25%           -0.33
50%           -0.18
75%           -0.03
max            1.25


In [20]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_generic_df,
    df_name = 'Generic',
    feature_cols = generic_features,
    target_cols = generic_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

### Forward Data Frame Analysis

In [21]:
# Define features and targets

forward_features = [
    'price', 'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

forward_targets = [
    'f_pct_px_chg_w1_b2', 'f_pct_px_chg_w2_b2', 'f_pct_px_chg_w3_b2', 
    'f_pct_px_chg_w4_b2', 'f_pct_px_chg_w5_b2', 'f_pct_px_chg_w6_b2',
    'f_pct_px_chg_w1_b4', 'f_pct_px_chg_w2_b4', 'f_pct_px_chg_w3_b4', 
    'f_pct_px_chg_w4_b4', 'f_pct_px_chg_w5_b4', 'f_pct_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
laa_forward_df = laa_forward_df[forward_targets + forward_features].dropna(subset=forward_targets)

laa_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 311 entries, 0 to 351
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   f_pct_px_chg_w1_b2  311 non-null    float64
 1   f_pct_px_chg_w2_b2  311 non-null    float64
 2   f_pct_px_chg_w3_b2  311 non-null    float64
 3   f_pct_px_chg_w4_b2  311 non-null    float64
 4   f_pct_px_chg_w5_b2  311 non-null    float64
 5   f_pct_px_chg_w6_b2  311 non-null    float64
 6   f_pct_px_chg_w1_b4  311 non-null    float64
 7   f_pct_px_chg_w2_b4  311 non-null    float64
 8   f_pct_px_chg_w3_b4  311 non-null    float64
 9   f_pct_px_chg_w4_b4  311 non-null    float64
 10  f_pct_px_chg_w5_b4  311 non-null    float64
 11  f_pct_px_chg_w6_b4  311 non-null    float64
 12  price               311 non-null    float64
 13  laa_homerun_dummy   311 non-null    int64  
 14  runners_on_base     311 non-null    float64
 15  inning              311 non-null    float64
 16  laa_score    

In [22]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_forward_df[laa_forward_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[forward_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_forward_df[laa_forward_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[forward_targets].describe(), 2))



LAA Home Runs:
       f_pct_px_chg_w1_b2  f_pct_px_chg_w2_b2  f_pct_px_chg_w3_b2  \
count              159.00              159.00              159.00   
mean                 0.19                0.23                0.24   
std                  0.29                0.32                0.31   
min                 -0.50               -0.61               -0.50   
25%                  0.02                0.04                0.04   
50%                  0.11                0.16                0.16   
75%                  0.29                0.31                0.34   
max                  1.50                2.00                1.50   

       f_pct_px_chg_w4_b2  f_pct_px_chg_w5_b2  f_pct_px_chg_w6_b2  \
count              159.00              159.00              159.00   
mean                 0.24                0.24                0.25   
std                  0.31                0.31                0.35   
min                 -0.50               -0.56               -0.56   
25%             

In [23]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_forward_df,
    df_name = 'Forward Binned Mean',
    feature_cols = forward_features,
    target_cols = forward_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

### Double Data Frame Analysis

In [24]:
# Define features and targets

double_features = [
    'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

double_targets = [
    'd_pct_px_chg_w1_b2', 'd_pct_px_chg_w2_b2', 'd_pct_px_chg_w3_b2', 
    'd_pct_px_chg_w4_b2', 'd_pct_px_chg_w5_b2', 'd_pct_px_chg_w6_b2',
    'd_pct_px_chg_w1_b4', 'd_pct_px_chg_w2_b4', 'd_pct_px_chg_w3_b4', 
    'd_pct_px_chg_w4_b4', 'd_pct_px_chg_w5_b4', 'd_pct_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
laa_double_df = laa_double_df[double_targets + double_features].dropna(subset=double_targets)

laa_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 294 entries, 0 to 351
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   d_pct_px_chg_w1_b2  294 non-null    float64
 1   d_pct_px_chg_w2_b2  294 non-null    float64
 2   d_pct_px_chg_w3_b2  294 non-null    float64
 3   d_pct_px_chg_w4_b2  294 non-null    float64
 4   d_pct_px_chg_w5_b2  294 non-null    float64
 5   d_pct_px_chg_w6_b2  294 non-null    float64
 6   d_pct_px_chg_w1_b4  294 non-null    float64
 7   d_pct_px_chg_w2_b4  294 non-null    float64
 8   d_pct_px_chg_w3_b4  294 non-null    float64
 9   d_pct_px_chg_w4_b4  294 non-null    float64
 10  d_pct_px_chg_w5_b4  294 non-null    float64
 11  d_pct_px_chg_w6_b4  294 non-null    float64
 12  laa_homerun_dummy   294 non-null    int64  
 13  runners_on_base     294 non-null    float64
 14  inning              294 non-null    float64
 15  laa_score           294 non-null    float64
 16  opp_score    

In [25]:
# General descriptive statistics

print("\n\nLAA Home Runs:")
laa_hr_df = laa_double_df[laa_double_df['laa_homerun_dummy'] == 1]
print(round(laa_hr_df[double_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = laa_double_df[laa_double_df['laa_homerun_dummy'] != 1]
print(round(opp_hr_df[double_targets].describe(), 2))



LAA Home Runs:
       d_pct_px_chg_w1_b2  d_pct_px_chg_w2_b2  d_pct_px_chg_w3_b2  \
count              150.00              150.00              150.00   
mean                 0.22                0.31                0.37   
std                  0.27                0.30                0.36   
min                 -0.25               -0.22               -0.15   
25%                  0.06                0.13                0.12   
50%                  0.15                0.24                0.29   
75%                  0.31                0.39                0.44   
max                  1.50                1.70                2.00   

       d_pct_px_chg_w4_b2  d_pct_px_chg_w5_b2  d_pct_px_chg_w6_b2  \
count              150.00              150.00              150.00   
mean                 0.44                0.45                0.48   
std                  0.52                0.49                0.54   
min                 -0.14               -0.11               -0.09   
25%             

In [26]:
# Correlation matrix

fig = plot_corr_matrix(
    df = laa_double_df,
    df_name = 'Double Binned Mean',
    feature_cols = double_features,
    target_cols = double_targets,
    team = 'LAA',
    team_colors = laa_colors
)

fig.show()

## Statistical Analysis for LAD

In [27]:
# Load in feature engineered data frames

lad_generic_df = pd.read_parquet("../data/processed/lad_generic_data.parquet")
lad_forward_df = pd.read_parquet("../data/processed/lad_forward_data.parquet")
lad_double_df = pd.read_parquet("../data/processed/lad_double_data.parquet")

### Generic Data Frame Analysis

In [28]:
# Define features and targets

generic_features = [
    'price', 'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

generic_targets = ['g_pct_px_chg']

# Look at a subset of the features, and drop any price change data if missing
lad_generic_df = lad_generic_df[generic_targets + generic_features].dropna(subset=['g_pct_px_chg'])

lad_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 337 entries, 0 to 346
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_pct_px_chg       337 non-null    float64
 1   price              337 non-null    float64
 2   lad_homerun_dummy  337 non-null    int64  
 3   runners_on_base    337 non-null    float64
 4   inning             337 non-null    float64
 5   lad_score          337 non-null    float64
 6   opp_score          337 non-null    float64
 7   score_delta        337 non-null    float64
 8   score_delta_abs    337 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 26.3 KB


In [29]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_generic_df[lad_generic_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[generic_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_generic_df[lad_generic_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[generic_targets].describe(), 2))



LAD Home Runs:
       g_pct_px_chg
count        191.00
mean           0.11
std            0.25
min           -0.29
25%            0.00
50%            0.02
75%            0.11
max            1.50

Opponent Home Runs:
       g_pct_px_chg
count        146.00
mean           0.14
std            2.73
min           -0.84
25%           -0.22
50%           -0.06
75%            0.00
max           32.50


In [30]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_generic_df,
    df_name = 'Generic',
    feature_cols = generic_features,
    target_cols = generic_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()

### Forward Data Frame Analysis

In [31]:
# Define features and targets

forward_features = [
    'price', 'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

forward_targets = [
    'f_pct_px_chg_w1_b2', 'f_pct_px_chg_w2_b2', 'f_pct_px_chg_w3_b2', 
    'f_pct_px_chg_w4_b2', 'f_pct_px_chg_w5_b2', 'f_pct_px_chg_w6_b2',
    'f_pct_px_chg_w1_b4', 'f_pct_px_chg_w2_b4', 'f_pct_px_chg_w3_b4', 
    'f_pct_px_chg_w4_b4', 'f_pct_px_chg_w5_b4', 'f_pct_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
lad_forward_df = lad_forward_df[forward_targets + forward_features].dropna(subset=forward_targets)

lad_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 317 entries, 0 to 346
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   f_pct_px_chg_w1_b2  317 non-null    float64
 1   f_pct_px_chg_w2_b2  317 non-null    float64
 2   f_pct_px_chg_w3_b2  317 non-null    float64
 3   f_pct_px_chg_w4_b2  317 non-null    float64
 4   f_pct_px_chg_w5_b2  317 non-null    float64
 5   f_pct_px_chg_w6_b2  317 non-null    float64
 6   f_pct_px_chg_w1_b4  317 non-null    float64
 7   f_pct_px_chg_w2_b4  317 non-null    float64
 8   f_pct_px_chg_w3_b4  317 non-null    float64
 9   f_pct_px_chg_w4_b4  317 non-null    float64
 10  f_pct_px_chg_w5_b4  317 non-null    float64
 11  f_pct_px_chg_w6_b4  317 non-null    float64
 12  price               317 non-null    float64
 13  lad_homerun_dummy   317 non-null    int64  
 14  runners_on_base     317 non-null    float64
 15  inning              317 non-null    float64
 16  lad_score    

In [32]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_forward_df[lad_forward_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[forward_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_forward_df[lad_forward_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[forward_targets].describe(), 2))



LAD Home Runs:
       f_pct_px_chg_w1_b2  f_pct_px_chg_w2_b2  f_pct_px_chg_w3_b2  \
count              182.00              182.00              182.00   
mean                 0.13                0.15                0.16   
std                  0.27                0.29                0.30   
min                 -0.50               -0.46               -0.18   
25%                  0.00                0.01                0.01   
50%                  0.04                0.05                0.06   
75%                  0.13                0.15                0.16   
max                  1.56                1.85                1.86   

       f_pct_px_chg_w4_b2  f_pct_px_chg_w5_b2  f_pct_px_chg_w6_b2  \
count              182.00              182.00              182.00   
mean                 0.17                0.18                0.18   
std                  0.35                0.40                0.38   
min                 -0.46               -0.61               -0.64   
25%             

In [33]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_forward_df,
    df_name = 'Forward Binned Mean',
    feature_cols = forward_features,
    target_cols = forward_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()

### Double Data Frame Analysis

In [34]:
# Define features and targets

double_features = [
    'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score',
    'score_delta', 'score_delta_abs'
]

double_targets = [
    'd_pct_px_chg_w1_b2', 'd_pct_px_chg_w2_b2', 'd_pct_px_chg_w3_b2', 
    'd_pct_px_chg_w4_b2', 'd_pct_px_chg_w5_b2', 'd_pct_px_chg_w6_b2',
    'd_pct_px_chg_w1_b4', 'd_pct_px_chg_w2_b4', 'd_pct_px_chg_w3_b4', 
    'd_pct_px_chg_w4_b4', 'd_pct_px_chg_w5_b4', 'd_pct_px_chg_w6_b4',
]

# Look at a subset of the features, and drop any price change data if missing
lad_double_df = lad_double_df[double_targets + double_features].dropna(subset=double_targets)

lad_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 305 entries, 0 to 346
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   d_pct_px_chg_w1_b2  305 non-null    float64
 1   d_pct_px_chg_w2_b2  305 non-null    float64
 2   d_pct_px_chg_w3_b2  305 non-null    float64
 3   d_pct_px_chg_w4_b2  305 non-null    float64
 4   d_pct_px_chg_w5_b2  305 non-null    float64
 5   d_pct_px_chg_w6_b2  305 non-null    float64
 6   d_pct_px_chg_w1_b4  305 non-null    float64
 7   d_pct_px_chg_w2_b4  305 non-null    float64
 8   d_pct_px_chg_w3_b4  305 non-null    float64
 9   d_pct_px_chg_w4_b4  305 non-null    float64
 10  d_pct_px_chg_w5_b4  305 non-null    float64
 11  d_pct_px_chg_w6_b4  305 non-null    float64
 12  lad_homerun_dummy   305 non-null    int64  
 13  runners_on_base     305 non-null    float64
 14  inning              305 non-null    float64
 15  lad_score           305 non-null    float64
 16  opp_score    

In [35]:
# General descriptive statistics

print("\n\nLAD Home Runs:")
lad_hr_df = lad_double_df[lad_double_df['lad_homerun_dummy'] == 1]
print(round(lad_hr_df[double_targets].describe(), 2))

print("\nOpponent Home Runs:")
opp_hr_df = lad_double_df[lad_double_df['lad_homerun_dummy'] != 1]
print(round(opp_hr_df[double_targets].describe(), 2))



LAD Home Runs:
       d_pct_px_chg_w1_b2  d_pct_px_chg_w2_b2  d_pct_px_chg_w3_b2  \
count              176.00              176.00              176.00   
mean                 0.13                0.17                0.22   
std                  0.24                0.25                0.29   
min                 -0.50               -0.33               -0.20   
25%                  0.01                0.03                0.05   
50%                  0.06                0.10                0.14   
75%                  0.12                0.23                0.26   
max                  1.25                1.22                1.80   

       d_pct_px_chg_w4_b2  d_pct_px_chg_w5_b2  d_pct_px_chg_w6_b2  \
count              176.00              176.00              176.00   
mean                 0.25                0.26                0.27   
std                  0.39                0.46                0.56   
min                 -0.04               -0.08               -0.44   
25%             

In [36]:
# Correlation matrix

fig = plot_corr_matrix(
    df = lad_double_df,
    df_name = 'Double Binned Mean',
    feature_cols = double_features,
    target_cols = double_targets,
    team = 'LAD',
    team_colors = lad_colors
)

fig.show()